In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:

import tensorflow as tf
from src.utils.dataframe_utils import create_dataframe


df = create_dataframe("../data/train.csv", "../data/boneage-training-dataset", segmented=False)
df_val = create_dataframe("../data/validation.csv", "../data/boneage-validation-dataset", segmented=False)

In [10]:
from src.preprocessing.scaling import scaling_data
dataset, dataset_val, scaler = scaling_data(df, df_val)

In [ ]:
from src.preprocessing.datasets import load_image, create_dataset_tf

dataset = create_dataset_tf(dataset, load_image)
dataset_val = create_dataset_tf(dataset_val, load_image)


In [12]:
# define a simple CNN for regression (e.g. bone age)
from src.model.cnn import CNN
model = CNN()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 224, 224, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,993 (394.50 KB)

 Trainable params: 100,993 (394.50 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
#to tune 
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [14]:
history = model.fit(
    dataset,
    validation_data=dataset_val,
    epochs=100,
    callbacks=[early_stop]
)


Epoch 1/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 122s 305ms/step - loss: 0.9884 - mae: 0.8090 - val_loss: 1.0846 - val_mae: 0.8553
Epoch 2/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 122s 307ms/step - loss: 0.9121 - mae: 0.7712 - val_loss: 1.1438 - val_mae: 0.8689
Epoch 3/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 122s 308ms/step - loss: 0.7770 - mae: 0.6979 - val_loss: 1.2927 - val_mae: 0.9137
Epoch 4/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 123s 310ms/step - loss: 0.7439 - mae: 0.6821 - val_loss: 1.0795 - val_mae: 0.8241
Epoch 5/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 122s 308ms/step - loss: 0.7312 - mae: 0.6733 - val_loss: 1.0790 - val_mae: 0.8344
Epoch 6/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 123s 310ms/step - loss: 0.7220 - mae: 0.6684 - val_loss: 1.0505 - val_mae: 0.8121
Epoch 7/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 128s 323ms/step - loss: 0.7024 - mae: 0.6585 - val_loss: 1.0918 - val_mae: 0.8367
Epoch 8/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 122s 309ms/step - loss: 0.6945 - mae: 0.6549 - val_loss: 1.0126 - val_mae: 0.8007
Epoch 9/100
395/

In [ ]:
import pickle

with open("model_results/training_history.pkl", "wb") as f:
    pickle.dump(history.history, f)

model.save("model_results/modello.keras")
